In [3]:
import numpy as np
import pandas as pd

#  Bayes' Theorem 

Formula:

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

Where:


$P(A)$ is the Prior probability.


$P(B|A)$ is the Likelihood.


$P(B)$ is the Evidence (total probability of $B$).


$P(A|B)$ is the Posterior probability.

## Task 1: General Bayes' Posterior Function
Write a Python function `bayes_posterior(prior, likelihood, evidence)` that calculates and returns the posterior probability using the Bayes' theorem formula for given values.

In [4]:
def bayes_posterior(prior: float, likelihood: float, evidence: float) -> float:
    """
    Calculates the posterior probability using Bayes' Theorem.
    
    Parameters:
    - prior (float): P(A), initial probability of event A.
    - likelihood (float): P(B|A), probability of evidence B given event A occurred.
    - evidence (float): P(B), total probability of evidence B occurring.
    
    Returns:
    - float: P(A|B), posterior probability of event A given evidence B.
    """
    if evidence == 0:
        raise ValueError("Evidence probability P(B) cannot be zero.")
    
    posterior = (likelihood * prior) / evidence
    return posterior

# Quick test with dummy values
print("Sample Posterior:", bayes_posterior(prior=0.01, likelihood=0.9, evidence=0.05))

Sample Posterior: 0.18000000000000002


## Task 2: Swiggy Restaurant Classification
**Scenario:**
- 10% of restaurants on Swiggy are 'premium' P(Premium) = 0.10.
- 80% of premium restaurants offer a discount P(Discount | Premium) = 0.80.
- 20% of regular restaurants offer a discount P(Discount | Regular) = 0.20.

**Goal:** If a restaurant offers a discount, calculate the probability that it is premium P(Premium | Discount).

In [5]:
# 1. Define Prior Probabilities
# P(Premium): Probability that a randomly selected restaurant is premium
prior_premium = 0.10
# P(Regular): Probability that a restaurant is regular
prior_regular = 1.0 - prior_premium

# 2. Define Likelihoods
# P(Discount | Premium): Probability of offering a discount given the restaurant is premium
likelihood_discount_given_premium = 0.80
# P(Discount | Regular): Probability of offering a discount given the restaurant is regular
likelihood_discount_given_regular = 0.20

# 3. Calculate Total Evidence P(Discount) using the Law of Total Probability
# P(Discount) = P(Discount | Premium) * P(Premium) + P(Discount | Regular) * P(Regular)
evidence_discount = (likelihood_discount_given_premium * prior_premium) + \
                    (likelihood_discount_given_regular * prior_regular)

# 4. Calculate Posterior Probability using bayes_posterior function
posterior_premium_given_discount = bayes_posterior(
    prior=prior_premium,
    likelihood=likelihood_discount_given_premium,
    evidence=evidence_discount
)

# Display results in a structured format using Pandas
task2_summary = pd.DataFrame({
    "Component": ["Prior P(Premium)", "Likelihood P(Discount|Premium)", "Total Evidence P(Discount)", "Posterior P(Premium|Discount)"],
    "Probability": [prior_premium, likelihood_discount_given_premium, evidence_discount, posterior_premium_given_discount]
})

print(task2_summary.to_string(index=False))
print(f"\nProbability that a restaurant is premium given it has a discount: {posterior_premium_given_discount:.4f} ({posterior_premium_given_discount * 100:.2f}%)")

                     Component  Probability
              Prior P(Premium)     0.100000
Likelihood P(Discount|Premium)     0.800000
    Total Evidence P(Discount)     0.260000
 Posterior P(Premium|Discount)     0.307692

Probability that a restaurant is premium given it has a discount: 0.3077 (30.77%)


## Task 3: Email Spam Detection Filter Simulation
**Objective:** 
Given a dataset of 20 email subjects labeled as spam or non-spam (ham), compute the probability that an email is spam if it contains the word **"free"** using Bayes' Theorem:

$$P(\text{Spam} | \text{"free"}) = \frac{P(\text{"free"} | \text{Spam}) \cdot P(\text{Spam})}{P(\text{"free"})}$$

In [6]:
# Create a simulated dataset of 20 emails
email_data = {
    "subject": [
        "Claim your free voucher now", 
        "Meeting schedule for tomorrow", 
        "Get free spins and win cash", 
        "Project delivery timeline update", 
        "Win a free iPhone today", 
        "Quarterly budget report attached", 
        "Exclusive free trial for you", 
        "Lunch plans this Friday?", 
        "Earn free Bitcoin with one click", 
        "Notes from client sync", 
        "Free consultation with our expert", 
        "Can you review this pull request?", 
        "Hot discount with free shipping", 
        "Action required: update your password", 
        "Urgent: Claim your free gift card", 
        "Team outing details next week", 
        "Risk-free guaranteed investment", 
        "Invoice for last month services", 
        "Buy one get one free offer", 
        "HR policy updates for 2026"
    ],
    "is_spam": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
}

df_emails = pd.DataFrame(email_data)

# Feature engineering: Check if 'free' exists in email subject (case-insensitive)
df_emails["contains_free"] = df_emails["subject"].str.lower().str.contains(r"\bfree\b", regex=True).astype(int)

# Total count
total_emails = len(df_emails)

# 1. Prior Probability: P(Spam) and P(Ham)
p_spam = np.mean(df_emails["is_spam"] == 1)
p_ham = np.mean(df_emails["is_spam"] == 0)

# 2. Likelihoods: P('free' | Spam) and P('free' | Ham)
spam_emails = df_emails[df_emails["is_spam"] == 1]
ham_emails = df_emails[df_emails["is_spam"] == 0]

p_free_given_spam = np.mean(spam_emails["contains_free"] == 1)
p_free_given_ham = np.mean(ham_emails["contains_free"] == 1)

# 3. Evidence: P('free')
p_free = (p_free_given_spam * p_spam) + (p_free_given_ham * p_ham)

# 4. Posterior Probability: P(Spam | 'free')
p_spam_given_free = bayes_posterior(prior=p_spam, likelihood=p_free_given_spam, evidence=p_free)

# Display dataset sample and calculated values
print("Dataset Sample:")
print(df_emails.head(6))

spam_summary = pd.DataFrame({
    "Metric": [
        "Prior P(Spam)", 
        "Prior P(Ham)", 
        "Likelihood P('free' | Spam)", 
        "Likelihood P('free' | Ham)", 
        "Total Evidence P('free')", 
        "Posterior P(Spam | 'free')"
    ],
    "Value": [
        p_spam, 
        p_ham, 
        p_free_given_spam, 
        p_free_given_ham, 
        p_free, 
        p_spam_given_free
    ]
})

print("\n--- Bayes Spam Calculation Summary ---")
print(spam_summary.to_string(index=False))
print(f"\nResult: An email containing 'free' has a {p_spam_given_free * 100:.2f}% probability of being Spam.")

Dataset Sample:
                            subject  is_spam  contains_free
0       Claim your free voucher now        1              1
1     Meeting schedule for tomorrow        0              0
2       Get free spins and win cash        1              1
3  Project delivery timeline update        0              0
4           Win a free iPhone today        1              1
5  Quarterly budget report attached        0              0

--- Bayes Spam Calculation Summary ---
                     Metric  Value
              Prior P(Spam)    0.5
               Prior P(Ham)    0.5
Likelihood P('free' | Spam)    1.0
 Likelihood P('free' | Ham)    0.0
   Total Evidence P('free')    0.5
 Posterior P(Spam | 'free')    1.0

Result: An email containing 'free' has a 100.00% probability of being Spam.


## Task 4: Real-World App Problem (Instagram Bot Detection)

### Prompt Used:
> *"Generate a realistic, real-world Bayes' Theorem problem related to Instagram for detecting bot accounts based on whether an account has a default profile picture. Provide the prior, likelihoods, and what needs to be solved."*

### Problem Formulation:
- **Scenario:** 
  - On Instagram, **15%** of all accounts are bot accounts P(Bot) = 0.15.
  - **70%** of bot accounts do not have a custom profile picture (default avatar) P(NoPic | Bot) = 0.70.
  - Only **5%** of genuine accounts do not have a custom profile picture P(NoPic | Genuine) = 0.05.
- **Goal:** If a newly detected account has no profile picture, find the probability that it is a bot account P(Bot | NoPic).

In [7]:
# 1. Define Priors
p_bot = 0.15
p_genuine = 1.0 - p_bot

# 2. Define Likelihoods
p_nopic_given_bot = 0.70
p_nopic_given_genuine = 0.05

# 3. Compute Evidence: P(NoPic) using NumPy array operations
priors = np.array([p_bot, p_genuine])
likelihoods = np.array([p_nopic_given_bot, p_nopic_given_genuine])

p_nopic = np.sum(priors * likelihoods)

# 4. Compute Posterior: P(Bot | NoPic)
p_bot_given_nopic = bayes_posterior(prior=p_bot, likelihood=p_nopic_given_bot, evidence=p_nopic)

# Output summary table using Pandas
app_summary = pd.DataFrame({
    "Parameter": [
        "P(Bot) [Prior]", 
        "P(Genuine) [Prior]", 
        "P(NoPic | Bot) [Likelihood]", 
        "P(NoPic | Genuine) [Likelihood]", 
        "P(NoPic) [Evidence]", 
        "P(Bot | NoPic) [Posterior]"
    ],
    "Value": [
        p_bot, 
        p_genuine, 
        p_nopic_given_bot, 
        p_nopic_given_genuine, 
        p_nopic, 
        p_bot_given_nopic
    ]
})

print(app_summary.to_string(index=False))
print(f"\nFinal Probability that the account is a bot: {p_bot_given_nopic:.4f} ({p_bot_given_nopic * 100:.2f}%)")

                      Parameter    Value
                 P(Bot) [Prior] 0.150000
             P(Genuine) [Prior] 0.850000
    P(NoPic | Bot) [Likelihood] 0.700000
P(NoPic | Genuine) [Likelihood] 0.050000
            P(NoPic) [Evidence] 0.147500
     P(Bot | NoPic) [Posterior] 0.711864

Final Probability that the account is a bot: 0.7119 (71.19%)
